In [8]:
import pandas as pd

df = pd.read_csv("quality_data.csv")
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)
df["input_text"] = df["input_text"].str.lower()
df["label"] = df["category"] + "_" + df["subcategory"]
df = df[["input_text", "label", "narration", "mode", "type", "category", "subcategory"]]
df.head()

,input_text,label,narration,mode,type,category,subcategory
0,narration: interest from fixed deposit | mode:...,investment_fd_interest,interest from fixed deposit,NEFT,Credit,investment,fd_interest
1,narration: fd interest payout | mode: neft | t...,investment_fd_interest,FD interest payout,NEFT,Credit,investment,fd_interest
2,narration: fixed deposit interest | mode: neft...,investment_fd_interest,Fixed deposit interest,NEFT,Credit,investment,fd_interest
3,narration: bank interest received | mode: neft...,investment_fd_interest,bank interest received,NEFT,Credit,investment,fd_interest
4,narration: fd maturity interest | mode: neft |...,investment_fd_interest,FD maturity interest,NEFT,Credit,investment,fd_interest


In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    df["input_text"],
    df["label"],
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]
)

In [11]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

In [12]:
from gliner import GLiNER

# GLiNER2 model
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")

labels = list(set(y_train))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\traitlets

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [13]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        best = max(entities, key=lambda x: x["score"])
        return best["label"], best["score"]

    return "unknown_unknown", 0.0

In [14]:
all_y_true = []
all_y_pred = []
all_conf = []   

for train_idx, test_idx in skf.split(df["input_text"], df["label"]):

    X_test_cv = df["input_text"].iloc[test_idx]
    y_test_cv = df["label"].iloc[test_idx]

    labels = list(set(df["label"].iloc[train_idx]))

    for text, actual in zip(X_test_cv, y_test_cv):
        pred, score = gliner_predict(text)   # 👈 capture score

        all_y_true.append(actual)
        all_y_pred.append(pred)
        all_conf.append(score)              # 👈 store it

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [15]:
result_df = df.copy().reset_index(drop=True)

result_df["predicted_label"] = all_y_pred
result_df["confidence"] = all_conf

In [16]:
print(len(df))
print(len(all_y_pred))
print(len(all_conf))

91
91
91


In [17]:
result_df = df.copy().reset_index(drop=True)
result_df["predicted_label"] = all_y_pred

In [18]:
result_df = df.copy().reset_index(drop=True)

result_df["predicted_label"] = all_y_pred
result_df["confidence"] = all_conf   # 👈 THIS IS MISSING IN YOUR CODE

def confidence_level(c):
    if c >= 0.6:
        return "High"
    elif c >= 0.3:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [19]:
# Actual split
result_df[["category", "subcategory"]] = result_df["label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [20]:
result_df["correct"] = result_df["label"] == result_df["predicted_label"]

In [21]:
print(result_df.columns)

Index(['input_text', 'label', 'narration', 'mode', 'type', 'category', 'subcategory', 'predicted_label', 'confidence', 'confidence_level', 'predicted_category', 'predicted_subcategory', 'correct'], dtype='object')


In [22]:
result_df = df_test.copy()

# sort back using original index
result_df = result_df.sort_index().reset_index(drop=True)

In [32]:
result_df = df.copy().reset_index(drop=True)

In [35]:
result_df["predicted_label"] = all_y_pred
result_df["confidence"] = all_conf

In [36]:
print(result_df.columns)

Index(['input_text', 'label', 'narration', 'mode', 'type', 'category', 'subcategory', 'confidence', 'predicted_label'], dtype='object')


In [37]:
# Add confidence
result_df["confidence"] = all_conf

# Split predicted label
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

# Correct flag
result_df["correct"] = result_df["label"] == result_df["predicted_label"]

In [38]:
final_df = result_df[[
    "narration",
    "category",
    "subcategory",
    "predicted_category",
    "predicted_subcategory",
    "confidence",
    "correct"
]]

final_df.head(20)

,narration,category,subcategory,predicted_category,predicted_subcategory,confidence,correct
0,interest from fixed deposit,investment,fd_interest,investment,fd_interest,0.780805,True
1,FD interest payout,investment,fd_interest,investment,fd_interest,0.695548,True
2,Fixed deposit interest,investment,fd_interest,investment,fd_booking,0.606321,False
3,bank interest received,investment,fd_interest,investment,fd_booking,0.815393,False
4,FD maturity interest,investment,fd_interest,investment,stocks,0.731849,False
5,Bank FD interest credit,investment,fd_interest,investment,stocks,0.916916,False
6,FD interest received,investment,fd_interest,income,salary,0.732133,False
7,roi amount credited,investment,fd_interest,income,salary,0.592414,False
8,Term deposit interest,investment,fd_interest,expense,food,0.588750,False
9,FD interest deposit,investment,fd_interest,expense,shopping,0.938736,False


In [39]:
from sklearn.metrics import classification_report
import numpy as np
report = classification_report(all_y_true, all_y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose().round(3)
# F2 Score
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)
report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)
report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                        precision  recall  f1-score  support  f2_score
expense_bills               0.750   0.750     0.750    8.000     0.750
expense_food                1.000   0.500     0.667    8.000     0.556
expense_health              0.600   0.429     0.500    7.000     0.455
expense_insurance           0.462   0.750     0.571    8.000     0.667
expense_loan                0.000   0.000     0.000    8.000     0.000
expense_shopping            0.727   1.000     0.842    8.000     0.930
expense_transport           0.750   0.900     0.818   10.000     0.865
income_salary               0.700   0.875     0.778    8.000     0.833
investment_fd_booking       0.727   1.000     0.842    8.000     0.930
investment_fd_interest      1.000   0.800     0.889   10.000     0.833
investment_stocks           0.889   1.000     0.941    8.000     0.976
accuracy                    0.736   0.736     0.736    0.736     0.736
macro avg                   0.691   0.728     0.691  

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [40]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(all_y_true, all_y_pred)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.736
Macro F1: 0.691
Weighted F1: 0.7


In [41]:
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
investment_stocks 0.941

Worst Performing Category:
expense_loan 0.0
